# Tutorial: Layer 5 Failure Risk Model


## 1. 목적

            5번 레이어는 **실패 리스크 모델**이다.

            핵심 질문은 이렇다.

            > 좋아 보이는 후보가 왜 실패할 수 있는가?

            ## 사용한 모델/방법

            - risk-band model: 정확한 실패 확률 대신 위험 구간으로 분류
            - translation uncertainty propagation: 4번 모델의 불확실성을 리스크에 반영
            - medical/contract/source/manual gates: 숫자로 잡히지 않는 실패 원인을 별도 게이트로 관리
            - locked recalibration: 후보명과 점수를 공개하지 않고 슬롯/리스크 밴드만 공개


## 2. 왜 확률 대신 risk band인가

            외국인 선수 데이터는 표본이 작고, 계약/부상/적응 같은 숨은 변수가 많다.

            그래서 "실패확률 37.2%"처럼 보이는 숫자는 오히려 위험하다.

            이 프로젝트에서는 다음처럼 더 방어 가능한 표현을 쓴다.

            - risk_screen_pass
            - watch_source_context
            - manual_review_required
            - block_until_source_or_medical_cleared


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent

OUT = ROOT / "outputs" / "tables"

def read_table(filename):
    path = OUT / filename
    if not path.exists():
        print(f"[missing] {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

def take_cols(df, cols, n=10):
    keep = [c for c in cols if c in df.columns]
    if not keep:
        return df.head(n)
    return df.loc[:, keep].head(n)

def count_by(df, cols):
    keep = [c for c in cols if c in df.columns]
    if not keep or df.empty:
        return pd.DataFrame()
    return df.groupby(keep, dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False)

def assert_candidate_names_locked(df):
    sensitive_cols = {"player_name", "search_name", "team_or_org", "player_id"}
    exposed = sensitive_cols.intersection(df.columns)
    if exposed:
        print(f"Candidate-sensitive columns exist but are not displayed here: {sorted(exposed)}")


In [ ]:
gate = read_table("recruitment_gate_status_v33.csv")
take_cols(gate[gate["gate"].eq("G5")], ["gate", "layer", "progress_pct", "status", "decision", "blocking_gap"])


In [ ]:
audit = read_table("layer5_6_augmented_recalibration_gate_audit_v0_1.csv")
take_cols(audit, ["gate", "layer", "check", "pass_rows", "total_rows", "status", "blocking_gap"], n=20)


## 3. 슬롯별 실패 리스크 분포

            아래 표는 후보명을 숨긴 상태에서 슬롯별 리스크 밴드만 보여준다.


In [ ]:
risk_summary = read_table("layer5_failure_risk_v0_3_slot_summary_v0_1.csv")
take_cols(risk_summary, ["fit_slot", "failure_risk_band_v0_3", "manual_review_tier_v0_3", "locked_rows", "release_allowed"], n=20)


In [ ]:
pivot = risk_summary.pivot_table(
    index="fit_slot",
    columns="failure_risk_band_v0_3",
    values="locked_rows",
    aggfunc="sum",
    fill_value=0,
)
pivot


## 4. 잠금된 리스크 원장

            실제 후보별 리스크 원장은 존재하지만, 발표 전에는 후보명/정확한 점수/순위를 공개하지 않는다.


In [ ]:
locked = read_table("layer5_failure_risk_v0_3_locked_recalibration_v0_1.csv")
assert_candidate_names_locked(locked)
count_by(locked, ["fit_slot", "failure_risk_band_v0_3", "manual_review_tier_v0_3"]).head(30)


## 5. 발표용 한 줄

            5번 레이어는 좋은 후보를 찾기 전에, 실패할 수 있는 경로를 계약/메디컬/번역불확실성/출처 부족으로 나눠 잠근 단계다.

            ## 연습문제

            한 후보가 statistical fit은 높지만 medical source가 비어 있다면, 왜 rank를 바로 공개하면 안 되는지 설명해보자.
